# Judge Error Statistics

This notebook summarizes error rows across three judge output CSV files. It checks `final_error`, malformed `map_results`, missing evidence-level results, and missing verdict fields.

## Setup

Update `MODEL_FILES` if you want to compare different judge runs.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "judge":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_FILES = {
    "gemini-2.5-flash": PROJECT_ROOT / "judge" / "judge_outputs_openrouter" / "factchecking_final_results_gemini-2.5-flash__gemini-2.5-flash__semantic__clip_finetuned__reranker_1__full.csv",
    "gemini-2.5-flash-lite": PROJECT_ROOT / "judge" / "judge_outputs_openrouter_flash_lite" / "factchecking_final_results_gemini-2.5-flash-lite__gemini-2.5-flash__semantic__clip_finetuned__reranker_1__full.csv",
    "deepseek-v4-flash": PROJECT_ROOT / "judge" / "judge_outputs_openrouter_deepseek_v4_flash" / "factchecking_final_results_deepseek-v4-flash__gemini-2.5-flash__semantic__clip_finetuned__reranker_1__full.csv",
}

OUTPUT_DIR = PROJECT_ROOT / "judge" / "judge_error_statistics_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_rows", 100)

MODEL_FILES

## Load CSV Files

In [ ]:
def load_model_outputs(model_files):
    frames = []
    file_status = []

    for model, path in model_files.items():
        exists = path.exists()
        file_status.append({
            "model": model,
            "path": str(path),
            "exists": exists,
            "size_mb": round(path.stat().st_size / 1024 / 1024, 2) if exists else None,
        })
        if not exists:
            continue

        df = pd.read_csv(path)
        df["model"] = model
        df["source_file"] = str(path)
        frames.append(df)

    if not frames:
        raise FileNotFoundError("No configured CSV files were found. Update MODEL_FILES.")

    return pd.concat(frames, ignore_index=True), pd.DataFrame(file_status)


df_all, df_file_status = load_model_outputs(MODEL_FILES)
display(df_file_status)
display(df_all.head(3))
print(f"Loaded rows: {len(df_all):,}")

## Mark Error Types

`final_error` catches failures in the final verdict step. `map_results` can still contain `null` when an evidence-level call failed but the final verdict still returned.

In [ ]:
def parse_json_list(value):
    if pd.isna(value):
        return [], "missing"
    text = str(value).strip()
    if not text:
        return [], "empty"
    try:
        parsed = json.loads(text)
    except Exception as exc:
        return [], f"parse_error: {type(exc).__name__}: {exc}"
    if not isinstance(parsed, list):
        return [], f"not_list: {type(parsed).__name__}"
    return parsed, ""


def classify_final_error(error_text):
    text = "" if pd.isna(error_text) else str(error_text).strip()
    if not text:
        return "none"
    lower = text.lower()
    code_match = re.search(r"error code:\s*(\d+)|'code':\s*(\d+)|\"code\":\s*(\d+)", text, flags=re.IGNORECASE)
    if code_match:
        return f"http_{next(g for g in code_match.groups() if g)}"
    if "validation" in lower or "json" in lower or "schema" in lower:
        return "format_or_validation"
    if "rate limit" in lower or "rate_limit" in lower:
        return "rate_limit"
    if "timeout" in lower:
        return "timeout"
    return "other"


df = df_all.copy()
for col in ["claim_id", "verdict", "explanation", "map_results", "final_error", "gold_label"]:
    if col not in df.columns:
        df[col] = pd.NA

parsed = df["map_results"].apply(parse_json_list)
df["map_results_parsed"] = parsed.apply(lambda item: item[0])
df["map_results_parse_error"] = parsed.apply(lambda item: item[1])
df["map_results_count"] = df["map_results_parsed"].apply(len)
df["map_results_null_count"] = df["map_results_parsed"].apply(lambda items: sum(item is None for item in items))
df["map_results_relation_missing_count"] = df["map_results_parsed"].apply(
    lambda items: sum(isinstance(item, dict) and not str(item.get("relation", "")).strip() for item in items)
)

df["final_error_text"] = df["final_error"].fillna("").astype(str).str.strip()
df["has_final_error"] = df["final_error_text"] != ""
df["final_error_type"] = df["final_error_text"].apply(classify_final_error)
df["has_map_parse_error"] = df["map_results_parse_error"].fillna("").astype(str).str.startswith(("parse_error", "not_list"))
df["has_evidence_null"] = df["map_results_null_count"] > 0
df["has_missing_verdict"] = df["verdict"].fillna("").astype(str).str.strip() == ""
df["has_any_error"] = df[["has_final_error", "has_map_parse_error", "has_evidence_null", "has_missing_verdict"]].any(axis=1)

df[["model", "claim_id", "verdict", "has_final_error", "has_evidence_null", "has_map_parse_error", "has_missing_verdict", "final_error_type"]].head()

## Summary By Model

In [ ]:
summary = (
    df.groupby("model", dropna=False)
    .agg(
        rows=("claim_id", "size"),
        unique_claims=("claim_id", "nunique"),
        final_error_rows=("has_final_error", "sum"),
        evidence_null_rows=("has_evidence_null", "sum"),
        map_parse_error_rows=("has_map_parse_error", "sum"),
        missing_verdict_rows=("has_missing_verdict", "sum"),
        any_error_rows=("has_any_error", "sum"),
    )
    .reset_index()
)

summary["success_rows"] = summary["rows"] - summary["any_error_rows"]
for col in ["final_error_rows", "evidence_null_rows", "map_parse_error_rows", "missing_verdict_rows", "any_error_rows", "success_rows"]:
    summary[col.replace("_rows", "_rate") if col.endswith("_rows") else f"{col}_rate"] = (summary[col] / summary["rows"]).round(4)

display(summary.sort_values(["any_error_rows", "final_error_rows"], ascending=False))

## Final Error Types

In [ ]:
final_error_types = (
    df[df["has_final_error"]]
    .groupby(["model", "final_error_type"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["model", "rows"], ascending=[True, False])
)

display(final_error_types)

sample_final_errors = (
    df[df["has_final_error"]]
    .loc[:, ["model", "claim_id", "verdict", "final_error_type", "final_error_text"]]
    .sort_values(["model", "final_error_type", "claim_id"])
    .head(50)
)
display(sample_final_errors)

## Evidence-Level Errors

In [ ]:
evidence_error_summary = (
    df.groupby("model", dropna=False)
    .agg(
        rows=("claim_id", "size"),
        rows_with_null_evidence=("has_evidence_null", "sum"),
        total_null_evidence_calls=("map_results_null_count", "sum"),
        rows_with_map_parse_error=("has_map_parse_error", "sum"),
        relation_missing_count=("map_results_relation_missing_count", "sum"),
    )
    .reset_index()
)
evidence_error_summary["rows_with_null_evidence_rate"] = (evidence_error_summary["rows_with_null_evidence"] / evidence_error_summary["rows"]).round(4)

display(evidence_error_summary.sort_values("rows_with_null_evidence", ascending=False))

sample_evidence_errors = (
    df[df["has_evidence_null"] | df["has_map_parse_error"]]
    .loc[:, ["model", "claim_id", "verdict", "map_results_count", "map_results_null_count", "map_results_parse_error", "final_error_text"]]
    .sort_values(["model", "claim_id"])
    .head(50)
)
display(sample_evidence_errors)

## Verdict Distribution

In [ ]:
verdict_counts = (
    df.assign(verdict_clean=df["verdict"].fillna("<missing>").astype(str).str.strip().replace("", "<missing>"))
    .groupby(["model", "verdict_clean"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
)
verdict_pivot = verdict_counts.pivot_table(index="model", columns="verdict_clean", values="rows", fill_value=0, aggfunc="sum")
display(verdict_pivot)

## Classification Metrics Excluding Error Rows

This follows the same label mapping as `judge_pipeline_evaluation.ipynb`, but removes rows with `final_error`, malformed `map_results`, null evidence results, or missing verdict before computing metrics. This prevents fallback `NEI` rows from inflating scores.

In [ ]:
LABELS = ["supported", "refuted", "nei"]
VERDICT_TO_LABEL = {"SUPPORTED": "supported", "REFUTED": "refuted", "NEI": "nei"}


def normalize_gold_label(value):
    text = "" if pd.isna(value) else str(value).strip().lower()
    return text if text in LABELS else ""


def normalize_pred_label(value):
    text = "" if pd.isna(value) else str(value).strip().upper()
    return VERDICT_TO_LABEL.get(text, "")


def classification_report_frame(y_true, y_pred, labels):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    rows = []
    total = len(y_true)

    for label in labels:
        tp = int(((y_true == label) & (y_pred == label)).sum())
        fp = int(((y_true != label) & (y_pred == label)).sum())
        fn = int(((y_true == label) & (y_pred != label)).sum())
        support = int((y_true == label).sum())
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({"label": label, "precision": precision, "recall": recall, "f1": f1, "support": support, "tp": tp, "fp": fp, "fn": fn})

    report = pd.DataFrame(rows)
    accuracy = float((y_true == y_pred).mean()) if total else 0.0
    macro = report[["precision", "recall", "f1"]].mean(numeric_only=True)
    weights = report["support"] / report["support"].sum() if report["support"].sum() else 0
    weighted = report[["precision", "recall", "f1"]].multiply(weights, axis=0).sum()

    summary_rows = pd.DataFrame([
        {"label": "accuracy", "precision": pd.NA, "recall": pd.NA, "f1": accuracy, "support": total, "tp": pd.NA, "fp": pd.NA, "fn": pd.NA},
        {"label": "macro_avg", "precision": macro["precision"], "recall": macro["recall"], "f1": macro["f1"], "support": total, "tp": pd.NA, "fp": pd.NA, "fn": pd.NA},
        {"label": "weighted_avg", "precision": weighted["precision"], "recall": weighted["recall"], "f1": weighted["f1"], "support": total, "tp": pd.NA, "fp": pd.NA, "fn": pd.NA},
    ])
    return pd.concat([report, summary_rows], ignore_index=True)


df_metrics_base = df.copy()
df_metrics_base["gold_label_norm"] = df_metrics_base["gold_label"].apply(normalize_gold_label)
df_metrics_base["pred_label"] = df_metrics_base["verdict"].apply(normalize_pred_label)
df_metrics_base["has_invalid_label"] = (df_metrics_base["gold_label_norm"] == "") | (df_metrics_base["pred_label"] == "")

df_eval_clean = df_metrics_base[(~df_metrics_base["has_any_error"]) & (~df_metrics_base["has_invalid_label"])].copy()

metrics_input_summary = (
    df_metrics_base.groupby("model", dropna=False)
    .agg(
        raw_rows=("claim_id", "size"),
        dropped_error_rows=("has_any_error", "sum"),
        invalid_label_rows=("has_invalid_label", "sum"),
    )
    .reset_index()
)
eligible_counts = df_eval_clean.groupby("model", dropna=False).size().rename("eligible_rows").reset_index()
metrics_input_summary = metrics_input_summary.merge(eligible_counts, on="model", how="left").fillna({"eligible_rows": 0})
metrics_input_summary["dropped_total_rows"] = metrics_input_summary["raw_rows"] - metrics_input_summary["eligible_rows"]
metrics_input_summary["eligible_rate"] = (metrics_input_summary["eligible_rows"] / metrics_input_summary["raw_rows"]).round(4)
display(metrics_input_summary)

print(f"Rows used for metrics after dropping errors/invalid labels: {len(df_eval_clean):,} / {len(df_metrics_base):,}")

In [ ]:
metric_reports = []
confusion_frames = []
misclassified_frames = []

for model, group in df_eval_clean.groupby("model", dropna=False):
    report = classification_report_frame(group["gold_label_norm"], group["pred_label"], LABELS)
    report.insert(0, "model", model)
    metric_reports.append(report)

    confusion = pd.crosstab(group["gold_label_norm"], group["pred_label"], rownames=["gold"], colnames=["pred"])
    confusion = confusion.reindex(index=LABELS, columns=LABELS, fill_value=0)
    confusion = confusion.reset_index().melt(id_vars="gold", var_name="pred", value_name="count")
    confusion.insert(0, "model", model)
    confusion_frames.append(confusion)

    misclassified = group[group["gold_label_norm"] != group["pred_label"]].copy()
    misclassified_frames.append(misclassified)

df_metric_report = pd.concat(metric_reports, ignore_index=True) if metric_reports else pd.DataFrame()
df_confusion_long = pd.concat(confusion_frames, ignore_index=True) if confusion_frames else pd.DataFrame()
df_misclassified_clean = pd.concat(misclassified_frames, ignore_index=True) if misclassified_frames else pd.DataFrame()

display(df_metric_report)

accuracy_table = (
    df_metric_report[df_metric_report["label"].isin(["accuracy", "macro_avg", "weighted_avg"])]
    .pivot(index="model", columns="label", values="f1")
    .reset_index()
)
display(accuracy_table)

In [ ]:
for model in df_eval_clean["model"].dropna().unique():
    print(f"\n=== {model} ===")
    confusion_pivot = (
        df_confusion_long[df_confusion_long["model"] == model]
        .pivot(index="gold", columns="pred", values="count")
        .reindex(index=LABELS, columns=LABELS, fill_value=0)
    )
    display(confusion_pivot)

if df_misclassified_clean.empty:
    print("No misclassified rows after dropping error rows.")
else:
    print(f"Misclassified clean rows: {len(df_misclassified_clean):,}")
    cols = ["model", "claim_id", "gold_label_norm", "pred_label", "verdict", "normalized_claim", "explanation"]
    cols = [col for col in cols if col in df_misclassified_clean.columns]
    display(df_misclassified_clean[cols].head(50))

## Claim Error Overlap

In [ ]:
claim_error_rows = df[df["has_any_error"]].assign(claim_id_str=lambda x: x["claim_id"].astype(str))
records = []
for claim_id_str, group in claim_error_rows.groupby("claim_id_str", dropna=False):
    records.append({
        "claim_id_str": claim_id_str,
        "error_models": sorted(group["model"].dropna().unique().tolist()),
        "error_model_count": group["model"].nunique(),
        "final_error_models": sorted(group.loc[group["has_final_error"], "model"].dropna().unique().tolist()),
        "evidence_null_models": sorted(group.loc[group["has_evidence_null"], "model"].dropna().unique().tolist()),
        "map_parse_error_models": sorted(group.loc[group["has_map_parse_error"], "model"].dropna().unique().tolist()),
    })

error_claims = pd.DataFrame(records).sort_values(["error_model_count", "claim_id_str"], ascending=[False, True])

display(error_claims.head(100))
print(f"Claims with any error in at least one model: {len(error_claims):,}")
print(f"Claims with errors in all configured models: {(error_claims['error_model_count'] == len(MODEL_FILES)).sum():,}")

## Export Reports

In [ ]:
error_rows = df[df["has_any_error"]].copy()
export_cols = [
    "model",
    "claim_id",
    "gold_label",
    "verdict",
    "has_final_error",
    "final_error_type",
    "final_error_text",
    "has_evidence_null",
    "map_results_null_count",
    "has_map_parse_error",
    "map_results_parse_error",
    "normalized_claim",
    "source_file",
]
export_cols = [col for col in export_cols if col in error_rows.columns]

summary_path = OUTPUT_DIR / "judge_error_summary_by_model.csv"
final_error_types_path = OUTPUT_DIR / "judge_final_error_types.csv"
evidence_error_summary_path = OUTPUT_DIR / "judge_evidence_error_summary.csv"
error_rows_path = OUTPUT_DIR / "judge_error_rows_combined.csv"
error_claims_path = OUTPUT_DIR / "judge_error_claim_overlap.csv"
metrics_input_summary_path = OUTPUT_DIR / "judge_metrics_input_summary_excluding_errors.csv"
metric_report_path = OUTPUT_DIR / "judge_classification_metrics_excluding_errors.csv"
confusion_path = OUTPUT_DIR / "judge_confusion_matrix_excluding_errors_long.csv"
misclassified_path = OUTPUT_DIR / "judge_misclassified_rows_excluding_errors.csv"

summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
final_error_types.to_csv(final_error_types_path, index=False, encoding="utf-8-sig")
evidence_error_summary.to_csv(evidence_error_summary_path, index=False, encoding="utf-8-sig")
error_rows[export_cols].to_csv(error_rows_path, index=False, encoding="utf-8-sig")
error_claims.to_csv(error_claims_path, index=False, encoding="utf-8-sig")
metrics_input_summary.to_csv(metrics_input_summary_path, index=False, encoding="utf-8-sig")
df_metric_report.to_csv(metric_report_path, index=False, encoding="utf-8-sig")
df_confusion_long.to_csv(confusion_path, index=False, encoding="utf-8-sig")
df_misclassified_clean.to_csv(misclassified_path, index=False, encoding="utf-8-sig")

print("Saved:")
for path in [summary_path, final_error_types_path, evidence_error_summary_path, error_rows_path, error_claims_path, metrics_input_summary_path, metric_report_path, confusion_path, misclassified_path]:
    print(f"- {path}")